In [ ]:
!pip install easyocr -q
import os
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import easyocr
from transformers import BertTokenizer, AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Preprocessing pipeline for images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


class_names = ['ADVE', 'Email', 'Form', 'Letter', 'Memo', 'News', 'Note', 'Report', 'Resume', 'Scientific']

In [ ]:
class MultimodalFusionModel(nn.Module):
    def __init__(self, num_classes, bert_model_name):
        super(MultimodalFusionModel, self).__init__()
       
        self.resnet = models.resnet18(weights=None)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        self.resnet_features = nn.Sequential(*list(self.resnet.children())[:-1])
        
        self.bert = AutoModel.from_pretrained(bert_model_name)
        
        # Combined Head
        self.classifier_head = nn.Sequential(
            nn.Linear(512 + 768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        img_feats = torch.flatten(self.resnet_features(image), 1)
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        text_feats = bert_outputs.pooler_output
        combined = torch.cat((img_feats, text_feats), dim=1)
        return self.classifier_head(combined)

In [ ]:
weights_path = "/content/drive/MyDrive/ColabNotebooks/document-classifier/models/trained_models/best_fusion_model.pth"

model = MultimodalFusionModel(len(class_names), 'bert-base-uncased').to(device)

if os.path.exists(weights_path):
    checkpoint = torch.load(weights_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    print("Architecture synced and weights loaded.")
else:
    print(f"ERROR: Model not found at {weights_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Architecture synced and weights loaded.


In [ ]:
def classify_document(img_path):
    image = Image.open(img_path).convert('RGB')
    img_tensor = transform(image).unsqueeze(0).to(device)

    result = reader.readtext(img_path, detail=0)
    raw_text = " ".join(result) if result else "[NO_TEXT]"

    inputs = tokenizer(
        raw_text, 
        return_tensors='pt', 
        max_length=512, 
        truncation=True, 
        padding='max_length'
    ).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor, inputs['input_ids'], inputs['attention_mask'])
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        conf, predicted = torch.max(probabilities, 1)
        
    return class_names[predicted.item()], conf.item(), raw_text

In [10]:
test_folder = "/content/drive/MyDrive/ColabNotebooks/document-classifier/data/test"

if os.path.exists(test_folder):
    files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"{'File Name':<30} | {'Prediction':<12} | {'Conf'}")
    print("-" * 60)
    
    for img_name in files:
        full_path = os.path.join(test_folder, img_name)
        try:
            label, score, _ = classify_document(full_path)
            print(f"{img_name:<30} | {label:<12} | {score*100:.1f}%")
        except Exception as e:
            print(f"Error on {img_name}: {e}")

File Name                      | Prediction   | Conf
------------------------------------------------------------
resumetest1.jpg                | Resume       | 58.1%
ltt1.png                       | Letter       | 42.9%
advertismentTest1.jpg          | ADVE         | 96.9%
advertismentTest2.jpg          | ADVE         | 95.1%
newspaperTest1.jpg             | ADVE         | 67.5%
newspaperTest2.jpg             | News         | 77.9%
scientificpaperTest1.png       | Scientific   | 76.5%
resumeTest2.jpg                | Resume       | 66.7%
noteTest2-modernNot1980s.png   | Report       | 55.3%
noteTest1-itisrealyhard.png    | Memo         | 17.4%
